In [4]:
"""
Build the zonal training table for the TFT.

Inputs:  nyiso_zonal_hourly.parquet      (11 zones + total, UTC, interval-start)
         weather_observed_zonal.parquet  (117 cols, UTC)
Output:  zonal_features.parquet          long format, one row per hour x zone

Conventions locked in here:
- Backbone is NYISO interval-start UTC. Weather at hour H pairs with demand
  consumed during H -> H+1 (the gate-2 convention fix).
- Lags/rolling stats computed within each zone (groupby), never across zones.
- Lean weather set (6 vars); full 13 remain available on disk for ablations.
"""

from pathlib import Path

import numpy as np
import pandas as pd

DEMAND_PATH = Path("nyiso_zonal_hourly.parquet")
WEATHER_PATH = Path("weather_observed_zonal.parquet")
OUT_PATH = Path("zonal_features.parquet")

# zone -> weather coordinate key (H and I reuse neighbors: Option A)
ZONE_WEATHER = {
    "WEST": "A_WEST", "GENESE": "B_GENESE", "CENTRL": "C_CENTRL",
    "NORTH": "D_NORTH", "MHK VL": "E_MHKVL", "CAPITL": "F_CAPITL",
    "HUD VL": "G_HUDVL", "MILLWD": "G_HUDVL", "DUNWOD": "J_NYC",
    "N.Y.C.": "J_NYC", "LONGIL": "K_LONGIL",
}

LEAN_VARS = [
    "temperature_2m", "apparent_temperature", "relative_humidity_2m",
    "wind_speed_10m", "shortwave_radiation", "cloud_cover",
]

# US federal holidays via pandas' built-in calendar
from pandas.tseries.holiday import USFederalHolidayCalendar


def build_calendar(index: pd.DatetimeIndex) -> pd.DataFrame:
    """Calendar features on the UTC backbone, derived from Eastern local time."""
    local = index.tz_localize("UTC").tz_convert("US/Eastern")
    cal = pd.DataFrame(index=index)
    cal["hour"] = local.hour
    cal["day_of_week"] = local.dayofweek
    cal["month"] = local.month
    cal["is_weekend"] = (local.dayofweek >= 5).astype(int)
    hol = USFederalHolidayCalendar().holidays(
        start=local.min().date(), end=local.max().date()
    )
    cal["is_holiday"] = pd.Series(local.date, index=index).isin(hol.date).astype(int)
    return cal


def main() -> None:
    demand = pd.read_parquet(DEMAND_PATH)
    weather = pd.read_parquet(WEATHER_PATH)

    zones = [z for z in ZONE_WEATHER if z in demand.columns]
    assert len(zones) == 11, f"Expected 11 zones, found {zones}"

    # weather starts 2013-01-01 00:00; demand starts 05:00 UTC same day.
    # Intersect the indexes so every row has both.
    common = demand.index.intersection(weather.index)
    demand = demand.loc[common]
    weather = weather.loc[common]
    calendar = build_calendar(common)

    print(f"Backbone: {common.min()} -> {common.max()} ({len(common):,} hours)")

    frames = []
    for zone in zones:
        wkey = ZONE_WEATHER[zone]
        df = pd.DataFrame(index=common)
        df["zone"] = zone
        df["demand"] = demand[zone]

        # zone-local weather (lean set)
        for v in LEAN_VARS:
            df[v] = weather[f"{v}__{wkey}"]

        # V-shape temperature deviation (validated in Iteration 1)
        df["temp_vshape"] = (df["temperature_2m"] - 14.0).abs()

        # calendar (same for all zones)
        for c in calendar.columns:
            df[c] = calendar[c]

        # lags and rolling stats - WITHIN this zone only
        df["demand_lag24"] = df["demand"].shift(24)
        df["demand_lag168"] = df["demand"].shift(168)
        df["demand_roll24_mean"] = df["demand"].rolling(24).mean()
        df["demand_roll168_mean"] = df["demand"].rolling(168).mean()
        df["demand_roll24_std"] = df["demand"].rolling(24).std()

        frames.append(df.reset_index().rename(columns={"index": "utc"}))

    long_df = pd.concat(frames, ignore_index=True)

    # drop warm-up rows (first 168h per zone have NaN lags)
    before = len(long_df)
    long_df = long_df.dropna(subset=["demand_lag168", "demand_roll168_mean"])
    print(f"Dropped {before - len(long_df):,} warm-up rows (168h x 11 zones)")

    # time index for pytorch-forecasting: hours since global start, per row
    t0 = long_df["utc"].min()
    long_df["time_idx"] = ((long_df["utc"] - t0).dt.total_seconds() // 3600).astype(int)

    # ------------------------- diagnostics -------------------------
    print("\n================ DIAGNOSTICS ================")
    print(f"Rows:            {len(long_df):,}  (expect ~1.29M)")
    print(f"Zones:           {long_df['zone'].nunique()}")
    print(f"Columns:         {len(long_df.columns)}")
    print(f"NaN cells:       {long_df.isna().sum().sum()}")
    print(f"time_idx range:  {long_df['time_idx'].min()} -> "
          f"{long_df['time_idx'].max()}")
    per_zone = long_df.groupby("zone")["demand"].agg(["mean", "min", "max"])
    print(f"\nPer-zone demand (MW):\n{per_zone.round(0).to_string()}")
    print("=============================================")

    long_df.to_parquet(OUT_PATH, index=False)
    print(f"\nSaved {OUT_PATH} ({OUT_PATH.stat().st_size / 1e6:.1f} MB)")


if __name__ == "__main__":
    main()

Backbone: 2013-01-01 05:00:00 -> 2026-05-31 23:00:00 (117,571 hours)


Dropped 1,848 warm-up rows (168h x 11 zones)

================ DIAGNOSTICS ================
Rows:            1,291,433  (expect ~1.29M)
Zones:           11
Columns:         21


NaN cells:       0
time_idx range:  0 -> 117402

Per-zone demand (MW):
          mean     min      max
zone                           
CAPITL  1353.0   486.0   2424.0
CENTRL  1786.0   733.0   3021.0
DUNWOD   676.0   238.0   1516.0
GENESE  1107.0   499.0   2103.0
HUD VL  1093.0   433.0   2367.0
LONGIL  2357.0  1144.0   5747.0
MHK VL   878.0   162.0   1559.0
MILLWD   321.0    56.0    721.0
N.Y.C.  5829.0  3604.0  11456.0
NORTH    588.0   260.0    965.0
WEST    1737.0  1027.0   2821.0



Saved zonal_features.parquet (66.8 MB)


In [2]:
import pandas as pd

df = pd.read_parquet("zonal_features.parquet")
neg = df[(df["zone"] == "MILLWD") & (df["demand"] < 0)]

print(f"Negative MILLWD hours: {len(neg)} "
      f"of {(df['zone'] == 'MILLWD').sum():,} "
      f"({100 * len(neg) / (df['zone'] == 'MILLWD').sum():.3f}%)")

if len(neg) > 0:
    neg = neg.copy()
    neg["year"] = neg["utc"].dt.year
    print("\nBy year:")
    print(neg["year"].value_counts().sort_index().to_string())
    print(f"\nWorst hours:")
    print(neg.nsmallest(5, "demand")[["utc", "demand"]].to_string(index=False))

# also check: any other zone with suspiciously low values relative to its mean?
print("\nMin as % of mean, all zones:")
stats = df.groupby("zone")["demand"].agg(["mean", "min"])
stats["min_pct_of_mean"] = (100 * stats["min"] / stats["mean"]).round(1)
print(stats["min_pct_of_mean"].sort_values().to_string())

Negative MILLWD hours: 10 of 117,403 (0.009%)

By year:
year
2014    10

Worst hours:
                utc  demand
2014-04-15 09:00:00  -342.9
2014-04-15 08:00:00  -335.7
2014-04-15 07:00:00  -314.0
2014-04-15 06:00:00  -295.9
2014-04-15 05:00:00  -279.8

Min as % of mean, all zones:
zone
MILLWD   -106.8
MHK VL     18.4
DUNWOD     35.2
CAPITL     35.9
HUD VL     39.6
CENTRL     41.1
NORTH      44.3
GENESE     45.1
LONGIL     48.5
WEST       59.1
N.Y.C.     61.8


In [3]:
import pandas as pd

# 1) patch the source zonal file
df = pd.read_parquet("nyiso_zonal_hourly.parquet")
zones = [c for c in df.columns if c != "NYISO_TOTAL"]

bad = df["MILLWD"] < 0
print(f"Patching {bad.sum()} negative MILLWD hours "
      f"({df.index[bad].min()} -> {df.index[bad].max()})")

df.loc[bad, "MILLWD"] = None
df["MILLWD"] = df["MILLWD"].interpolate(method="linear")
df["NYISO_TOTAL"] = df[zones].sum(axis=1)

assert (df["MILLWD"] >= 0).all()
df.to_parquet("nyiso_zonal_hourly.parquet")
print("Source patched and saved.")

Patching 10 negative MILLWD hours (2014-03-20 04:00:00 -> 2014-04-15 11:00:00)


Source patched and saved.


In [5]:
import pandas as pd

df = pd.read_csv("data/processed/tft_clean.csv", usecols=["ts", "split"])
df["ts"] = pd.to_datetime(df["ts"])

print(df.groupby("split")["ts"].agg(["min", "max", "count"]).to_string())

                      min                 max  count
split                                               
test  2024-01-23 08:00:00 2026-05-30 00:00:00  20584
train 2015-07-01 01:00:00 2024-01-23 07:00:00  75080
